# 3. Train the two models

Same setup as 2024: one model for `rechtsgebieden`, one for `procedures`. Both read the ruling text and can output several labels.

Base model: `GroNLP/bert-base-dutch-cased` (BERTje). Each label gets its own score; a label is kept if the score is at least 0.5.

Rare labels from `labels.json` are removed. Rows with no procedures are skipped for the procedures model.

Install the modelling packages first, once:

```
pip install scikit-learn transformers datasets accelerate
pip install torch --index-url https://download.pytorch.org/whl/cu121
```

Do not run the two `trainer.train()` cells until you are ready. They take hours on the 4 GB laptop GPU.

In [5]:
from pathlib import Path
import json
import torch
import numpy as np
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import f1_score, precision_score, recall_score
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EvalPrediction,
)

DATA = Path("../data/processed")
MODEL_NAME = "GroNLP/bert-base-dutch-cased"
MAX_LEN = 512
BATCH = 2
EPOCHS = 3
LR = 2e-5

label_info = json.loads((DATA / "labels.json").read_text(encoding="utf-8"))
RG_KEEP = label_info["rechtsgebieden_keep"]
PR_KEEP = label_info["procedures_keep"]
print("rechtsgebieden keep:", len(RG_KEEP))
print("procedures keep:", len(PR_KEEP))

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

rechtsgebieden keep: 20
procedures keep: 25
True
NVIDIA GeForce RTX 3050 Laptop GPU


Turn each row into text + the labels we keep. `drop_empty` is for the procedures task.

In [6]:
def load_task(path: Path, field: str, keep: list[str], drop_empty: bool) -> pd.DataFrame:
    frame = pd.read_json(path, lines=True)
    keep_set = set(keep)
    frame["target"] = frame[field].map(lambda xs: [x for x in xs if x in keep_set])
    if drop_empty:
        frame = frame[frame["target"].map(len) > 0]
    return frame[["ecli", "text", "target"]].reset_index(drop=True)


def encode(frame: pd.DataFrame, mlb: MultiLabelBinarizer) -> Dataset:
    labels = mlb.transform(frame["target"]).astype("float32")
    data = Dataset.from_dict({"text": frame["text"].tolist(), "labels": labels.tolist()})
    return data.map(
        lambda batch: tokenizer(batch["text"], truncation=True, padding="max_length", max_length=MAX_LEN),
        batched=True,
    )


def scores(pred: EvalPrediction) -> dict:
    logits = pred.predictions
    if isinstance(logits, tuple):
        logits = logits[0]
    probs = 1.0 / (1.0 + np.exp(-np.asarray(logits)))
    y_hat = (probs >= 0.5).astype(int)
    y = np.asarray(pred.label_ids)
    return {
        "f1_micro": f1_score(y, y_hat, average="micro", zero_division=0),
        "f1_macro": f1_score(y, y_hat, average="macro", zero_division=0),
        "precision_micro": precision_score(y, y_hat, average="micro", zero_division=0),
        "recall_micro": recall_score(y, y_hat, average="micro", zero_division=0),
    }

## Rechtsgebieden

Load train and val, drop rare labels, show how many rows and a few targets.

In [8]:
rg_train = load_task(DATA / "train.jsonl", "rechtsgebieden", RG_KEEP, drop_empty=True)
rg_val = load_task(DATA / "val.jsonl", "rechtsgebieden", RG_KEEP, drop_empty=True)
print("RG train:", len(rg_train), "val:", len(rg_val))
print(rg_train["target"].head(8).tolist())

RG train: 48251 val: 6012
[['Strafrecht'], ['Strafrecht'], ['Strafrecht'], ['Strafrecht'], ['Civiel recht'], ['Bestuursrecht', 'Socialezekerheidsrecht'], ['Strafrecht'], ['Bestuursrecht', 'Belastingrecht']]


Three steps: tokenize (Map, already done — skip it), create the Trainer with save every 1000 steps, then train.

In [33]:
rg_mlb = MultiLabelBinarizer(classes=RG_KEEP)
rg_mlb.fit([RG_KEEP])

rg_train_ds = encode(rg_train, rg_mlb)
rg_val_ds = encode(rg_val, rg_mlb)
print("tokenized train", len(rg_train_ds), "val", len(rg_val_ds))














Map: 100%|██████████| 6012/6012 [01:14<00:00, 81.00 examples/s]

tokenized train 48251 val 6012


In [45]:
rg_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(RG_KEEP),
    problem_type="multi_label_classification",
)

rg_args = TrainingArguments(
    output_dir="../models/rg",
    overwrite_output_dir=True,
    eval_strategy="epoch",
    save_strategy="steps",
    save_steps=1000,
    learning_rate=LR,
    per_device_train_batch_size=BATCH,
    per_device_eval_batch_size=BATCH,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    logging_steps=50,
    save_total_limit=2,
    load_best_model_at_end=False,
    fp16=True,
)

rg_trainer = Trainer(
    model=rg_model,
    args=rg_args,
    train_dataset=rg_train_ds,
    eval_dataset=rg_val_ds,
    compute_metrics=scores,
)
print("trainer ready, save every", rg_args.save_steps, "steps")

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at GroNLP/bert-base-dutch-cased and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


trainer ready, save every 1000 steps


In [ ]:
from transformers.trainer_utils import get_last_checkpoint

rg_ckpt = get_last_checkpoint("../models/rg")
print("resume from:", rg_ckpt)
rg_trainer.train(resume_from_checkpoint=rg_ckpt)
rg_trainer.save_model("../models/rg")
tokenizer.save_pretrained("../models/rg")
Path("../models/rg/labels.json").write_text(
    json.dumps(list(rg_mlb.classes_), ensure_ascii=False, indent=2), encoding="utf-8"
)
print("saved ../models/rg")

## Procedures

Load train and val, drop rare labels, show how many rows and a few targets.

In [7]:
pr_train = load_task(DATA / "train.jsonl", "procedures", PR_KEEP, drop_empty=True)
pr_val = load_task(DATA / "val.jsonl", "procedures", PR_KEEP, drop_empty=True)
print("PR train:", len(pr_train), "val:", len(pr_val))
print(pr_train["target"].head(8).tolist())

PR train: 45857 val: 5779
[['Eerste aanleg - enkelvoudig'], ['Eerste aanleg - meervoudig'], ['Hoger beroep'], ['Tussenuitspraak'], ['Hoger beroep'], ['Hoger beroep'], ['Hoger beroep'], ['Hoger beroep']]


Three steps: tokenize (Map), create the Trainer with save every 1000 steps, then train.

In [8]:
pr_mlb = MultiLabelBinarizer(classes=PR_KEEP)
pr_mlb.fit([PR_KEEP])

pr_train_ds = encode(pr_train, pr_mlb)
pr_val_ds = encode(pr_val, pr_mlb)
print("tokenized train", len(pr_train_ds), "val", len(pr_val_ds))

Map: 100%|██████████| 5779/5779 [00:35<00:00, 162.59 examples/s]

tokenized train 45857 val 5779


In [9]:
pr_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(PR_KEEP),
    problem_type="multi_label_classification",
)

pr_args = TrainingArguments(
    output_dir="../models/pr",
    overwrite_output_dir=True,
    eval_strategy="epoch",
    save_strategy="steps",
    save_steps=1000,
    learning_rate=LR,
    per_device_train_batch_size=BATCH,
    per_device_eval_batch_size=BATCH,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    logging_steps=50,
    save_total_limit=2,
    load_best_model_at_end=False,
    fp16=True,
)

pr_trainer = Trainer(
    model=pr_model,
    args=pr_args,
    train_dataset=pr_train_ds,
    eval_dataset=pr_val_ds,
    compute_metrics=scores,
)
print("trainer ready, save every", pr_args.save_steps, "steps")


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at GroNLP/bert-base-dutch-cased and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


trainer ready, save every 1000 steps


In [ ]:
from transformers.trainer_utils import get_last_checkpoint

pr_ckpt = get_last_checkpoint("../models/pr")
print("resume from:", pr_ckpt)
pr_trainer.train(resume_from_checkpoint=pr_ckpt)
pr_trainer.save_model("../models/pr")
tokenizer.save_pretrained("../models/pr")
Path("../models/pr/labels.json").write_text(
    json.dumps(list(pr_mlb.classes_), ensure_ascii=False, indent=2), encoding="utf-8"
)
print("saved ../models/pr")


: 